# Qualitative comparison: 4 models × 3 datasets

Same models as `scripts/eval_per_dataset.sh`. Two test tiles per dataset, every model on every tile:
**green = TP, blue = FP, red = FN**, with the tile IoU in each title.
Then all four models on the real image used in `infer_tile.ipynb`.

In [ ]:
import sys
sys.path.append("..")

import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import patches
from PIL import Image
from torchvision import transforms

from src.models.factory import build_model

In [ ]:
runs_path = "/home/antoniocorvino/Projects/BuildingsExtraction"
dataset_root = "/mnt/nas151/sar/Footprint/datasets"
out_dir = Path(runs_path) / "experiments" / "qualitative"
out_dir.mkdir(parents=True, exist_ok=True)

arch = "unetLL"
threshold = 0.5          # same threshold as eval_per_dataset.sh, so the tile IoU matches the table
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fill in the run folders of the three single-dataset models (the _bs32 runs)
MODELS = {
    "WHU only":   f"{runs_path}/runs/WHUBuildingDataset/<run_name>/best_model.pth",
    "Inria only": f"{runs_path}/runs/InriaBuildingDataset/<run_name>/best_model.pth",
    "Mass. only": f"{runs_path}/runs/MassachusettsBuildingDataset_s128/<run_name>/best_model.pth",
    "Merged":     f"{runs_path}/runs/Merged_WHU_Inria_Mass/unetLL_bce_dim256_n68500_bs32/best_model.pth",
}

# Dataset -> split. Use the same split as in the IoU table (e.g. "val" for Inria if its test has no GT)
DATASETS = {
    "WHU":           ("WHUBuildingDataset", "test"),
    "Inria":         ("InriaBuildingDataset", "test"),
    "Massachusetts": ("MassachusettsBuildingDataset", "test"),
}

# Tiles to show: None = pick automatically, or a list of tile names without extension
TILES = {"WHU": None, "Inria": None, "Massachusetts": None}
N_TILES = 2
SEED = 0

In [ ]:
transform = transforms.ToTensor()     # same scaling as training ([0, 255] -> [0, 1])

def load(path):
    m = build_model(arch).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    return m.eval()

models = {name: load(p) for name, p in MODELS.items()}
print("Loaded:", ", ".join(models))

@torch.no_grad()
def predict(model, img):
    x = transform(img).unsqueeze(0).to(device)
    return torch.sigmoid(model(x)).squeeze().cpu().numpy()

def tile_dirs(ds_key):
    name, split = DATASETS[ds_key]
    base = Path(dataset_root) / name / "tiles_256" / split
    return base / "images", base / "gt"

def pick_tiles(ds_key, n=N_TILES, seed=SEED, lo=0.15, hi=0.5, max_scan=3000):
    """Random tiles whose GT building fraction is between lo and hi:
    enough buildings to judge, not a tile that is all roof."""
    img_dir, gt_dir = tile_dirs(ds_key)
    names = sorted(p.stem for p in img_dir.iterdir() if p.suffix.lower() in {".tif", ".tiff", ".png"})
    rng = random.Random(seed)
    rng.shuffle(names)
    picked = []
    for stem in names[:max_scan]:
        gt_path = next(gt_dir.glob(stem + ".*"), None)
        if gt_path is None:
            continue
        frac = (np.array(Image.open(gt_path).convert("L")) > 0).mean()
        if lo <= frac <= hi:
            picked.append(stem)
            if len(picked) == n:
                break
    if len(picked) < n:
        print(f"[{ds_key}] only {len(picked)} tile(s) in the {lo}-{hi} building range")
    return picked

def load_tile(ds_key, stem):
    img_dir, gt_dir = tile_dirs(ds_key)
    img = Image.open(next(img_dir.glob(stem + ".*"))).convert("RGB")
    gt = np.array(Image.open(next(gt_dir.glob(stem + ".*"))).convert("L")) > 0
    return img, gt

def error_overlay(pred, gt):
    ov = np.zeros((*gt.shape, 4), dtype=np.float32)
    ov[pred & gt] = [0, 1, 0, 0.45]      # TP green
    ov[pred & ~gt] = [0, 0, 1, 0.45]     # FP blue
    ov[~pred & gt] = [1, 0, 0, 0.45]     # FN red
    return ov

def iou(pred, gt):
    union = (pred | gt).sum()
    return (pred & gt).sum() / union if union else 1.0

## Test tiles: every model on every dataset

In [ ]:
rows = []
for ds_key in DATASETS:
    stems = TILES[ds_key] or pick_tiles(ds_key)
    rows += [(ds_key, s) for s in stems]
print("Tiles:", rows)

cols = ["Image", "Ground truth"] + list(models)
fig, axes = plt.subplots(len(rows), len(cols), figsize=(3 * len(cols), 3.1 * len(rows)), squeeze=False)

for r, (ds_key, stem) in enumerate(rows):
    img, gt = load_tile(ds_key, stem)
    img_np = np.array(img)

    axes[r, 0].imshow(img_np)
    axes[r, 0].set_ylabel(f"{ds_key}\n{stem}", fontsize=10)
    axes[r, 1].imshow(img_np)
    gt_ov = np.zeros((*gt.shape, 4), dtype=np.float32)
    gt_ov[gt] = [1, 0.85, 0, 0.5]        # GT in yellow (red is reserved for FN)
    axes[r, 1].imshow(gt_ov)

    for c, (name, model) in enumerate(models.items(), start=2):
        pred = predict(model, img) > threshold
        axes[r, c].imshow(img_np)
        axes[r, c].imshow(error_overlay(pred, gt))
        axes[r, c].set_title(f"IoU {iou(pred, gt):.2f}", fontsize=11)

for c, name in enumerate(cols):
    axes[0, c].set_title(name if c < 2 else f"{name}\n{axes[0, c].get_title()}", fontsize=12, fontweight="bold")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

fig.legend(handles=[patches.Patch(color="green", label="True positive"),
                    patches.Patch(color="blue", label="False positive"),
                    patches.Patch(color="red", label="False negative")],
           loc="lower center", ncol=3, fontsize=12, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig(out_dir / "tiles_4models.png", dpi=200, bbox_inches="tight")
plt.show()

## Real image (no ground truth): all four models

The same image as `infer_tile.ipynb`. Probability maps, so the confidence of each model is visible.

In [ ]:
image_path = f"{runs_path}/experiments/thesis/testing1.png"
img = Image.open(image_path).convert("RGB")
img_np = np.array(img)

fig, axes = plt.subplots(1, len(models) + 1, figsize=(5 * (len(models) + 1), 4.2))
axes[0].imshow(img_np); axes[0].set_title("Image", fontsize=14, fontweight="bold")
for ax, (name, model) in zip(axes[1:], models.items()):
    prob = predict(model, img)
    ax.imshow(img_np)
    im = ax.imshow(prob, cmap="jet", alpha=0.5, vmin=0, vmax=1, interpolation="bilinear")
    ax.set_title(name, fontsize=14, fontweight="bold")
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.012, pad=0.01, label="Building probability")
plt.savefig(out_dir / "real_image_4models.png", dpi=200, bbox_inches="tight")
plt.show()